In [4]:
# ldpc_bsc_gpu.py
import math, numpy as np, torch

# ----------------------------
# 1) Make H in systematic form
# ----------------------------
def make_ldpc_systematic(K: int, R: float = None, M: int = None, col_w: int = 3, seed: int | None = 0):
    """
    Build a binary LDPC parity-check matrix in systematic form H = [A | I_M].
    - K: number of information bits
    - R: desired code rate (0<R<1). If given, M = round(K*(1-R)/R).
    - M: number of parity checks/rows (ignored if R is given)
    - col_w: column weight for A (regular construction, typical 3 or 4)
    Returns: H (np.uint8) of shape (M, K+M)
    """
    assert K > 0
    if R is not None:
        assert 0 < R < 1, "Rate R must be in (0,1)."
        M = max(1, int(round(K*(1-R)/R)))
    assert isinstance(M, int) and M > 0, "Provide a positive M or valid R."

    rng = np.random.default_rng(seed)
    # Build A (M x K) with exactly col_w ones per column, distributing across lightest rows
    rows = []
    cols = []
    row_load = np.zeros(M, dtype=int)
    for j in range(K):
        # pick 'col_w' distinct rows, preferring currently light rows
        order = np.argsort(row_load, kind='stable')
        pool = order[:min(M, 2*col_w + 8)]
        sel = rng.choice(pool, size=min(col_w, len(pool)), replace=False)
        rows.extend(sel.tolist())
        cols.extend([j]*len(sel))
        row_load[sel] += 1

    A = np.zeros((M, K), dtype=np.uint8)
    if len(rows) > 0:
        A[rows, cols] ^= 1  # place ones

    # ensure no all-zero rows
    zrows = np.where(A.sum(axis=1) == 0)[0]
    for r in zrows:
        j = rng.integers(0, K)
        A[r, j] ^= 1

    # form H = [A | I]
    H = np.zeros((M, K+M), dtype=np.uint8)
    H[:, :K] = A
    H[:, K:] = np.eye(M, dtype=np.uint8)
    return H

# ----------------------------
# 2) Encoder/Simulator/Decoder
# ----------------------------
def simulate_ldpc_bsc_fixedllr(H: np.ndarray, p: float, L0: float = 10.0,
                               nframes: int = 200, max_iter: int = 50,
                               use_gpu: bool = True):
    """
    LDPC over BSC(p) with fixed-magnitude LLRs (±L0) and a bit-flip decoder.
    H: np.uint8 (M x N) with right block = I_M  (systematic)
    p: crossover probability (pre-decoder BER)
    Returns: dict with BLER, BER, etc.
    """
    assert H.dtype == np.uint8 and H.ndim == 2
    M, N = H.shape
    K = N - M
    assert np.all(H[:, K:] == np.eye(M, dtype=np.uint8)), "H must be [A|I]."

    device = 'cuda' if (use_gpu and torch.cuda.is_available()) else 'cpu'
    H_t = torch.tensor(H, dtype=torch.float32, device=device)  # 0/1 matrix
    A_t = H_t[:, :K]
    I_M = H_t[:, K:]  # not used directly but available
    n_block_err = 0
    n_bit_err = 0

    for _ in range(nframes):
        # ---- Encode (systematic): p_bits = (A @ u) mod 2; c = [u; p_bits]
        u = torch.randint(0, 2, (K,), device=device, dtype=torch.float32)
        p_bits = (A_t @ u) % 2.0
        c = torch.cat([u, p_bits])  # length N, 0/1 floats

        # ---- BSC channel and fixed LLRs
        flips = (torch.rand(N, device=device) < p).float()
        r = (c + flips) % 2.0
        llr = L0 * (1.0 - 2.0 * r)   # 0 -> +L0, 1 -> -L0  (we keep llr for possible soft use)

        # ---- Bit-flip (Gallager-B) hard-decision decoder
        chat = (llr < 0).float()     # initial hard decision
        for _it in range(max_iter):
            synd = (H_t @ chat) % 2.0                  # M
            if synd.sum() == 0:
                break
            uns = H_t.t() @ synd                       # N, number of unsatisfied checks per bit
            thr = uns.max()
            flip_mask = (uns == thr) & (thr > 0)
            chat = (chat + flip_mask.float()) % 2.0    # flip those bits

        uhat = chat[:K]
        bit_err = (uhat != u).sum().item()
        n_bit_err += int(bit_err)
        n_block_err += int(bit_err > 0)

    return {
        'BLER': n_block_err / nframes,
        'BER' : n_bit_err / (nframes * K),
        'frames': nframes,
        'K': K, 'M': M, 'N': N, 'R': K/N,
        'device': device
    }

# ----------------------------
# 3) Example use cases
# ----------------------------
if __name__ == "__main__":
    # # Example A: you specify K and R
    # K, R = 512, 0.20          # 20% rate => heavy redundancy
    # H = make_ldpc_systematic(K, R=R, col_w=3, seed=42)
    # res = simulate_ldpc_bsc_fixedllr(H, p=0.30, L0=10, nframes=200, max_iter=100, use_gpu=True)
    # print("Example A:", res)

    # Example B: you specify K and M directly
    K, M = 256, 16000          # N=1500, R=0.2 (same rate via explicit M)
    H2 = make_ldpc_systematic(K, M=M, col_w=4, seed=1)
    res2 = simulate_ldpc_bsc_fixedllr(H2, p=0.30, L0=10, nframes=200, max_iter=100, use_gpu=True)
    print("Example B:", res2)


Example B: {'BLER': 0.895, 'BER': 0.00982421875, 'frames': 200, 'K': 256, 'M': 16000, 'N': 16256, 'R': 0.015748031496062992, 'device': 'cuda'}
